# 08f Slope + Safety Push Lane Postprocess v1

08d/08e에서 확인한 문제는 명확했다.

```text
decoder lane points -> target path 재구성 -> left/right anchor 판단
```

이 구조는 직관적으로는 좋아 보이지만, 실제로는 decoder x 좌표가 frame마다 흔들릴 때 변수가 너무 많아진다.  
특히 `left/right` 판단, offset path, anchor matching이 모두 x 좌표 안정성에 의존한다.

08f는 반대로 접근한다.

```text
lane point들
-> lane의 local slope는 방향 신호로 사용
-> lane이 화면 중앙 기준 너무 가까우면 안전 push로 밀어냄
-> 후처리는 steer 추천 + risk/confidence 진단만 출력
```

즉, 08f는 target path를 만들지 않는다.  
차선을 "따라갈 선"으로 재구성하지 않고, 차선이 알려주는 **방향(direction)** 과 **이탈 위험(departure risk)** 만 뽑는다.

이 구조의 목표는 다음이다.

- 08b처럼 기울기 기반의 단순성을 유지한다.
- 차선을 밟는 문제는 `safety_push`로 보완한다.
- 상태머신과 결합하기 쉽도록 `steer_norm`, `departure_risk`, `lane_confidence`를 분리해서 출력한다.

In [1]:
from __future__ import annotations

import csv
import json
from collections import Counter
from pathlib import Path

import cv2
import numpy as np


BASE = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild")
REVIEW_ROOT = BASE / "review_outputs" / "08f_slope_push_lane_postprocess_v1"
VIDEO_DIR = REVIEW_ROOT / "videos"
TABLE_DIR = REVIEW_ROOT / "tables"
CONFIG_DIR = REVIEW_ROOT / "config"
FRAME_REVIEW_DIR = REVIEW_ROOT / "frame_review"
for d in [REVIEW_ROOT, VIDEO_DIR, TABLE_DIR, CONFIG_DIR, FRAME_REVIEW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PKG10 = BASE / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1" / "pkg"
RECORDS_CSV = PKG10 / "t" / "records_manifest.csv"
DECODED_JSONL = PKG10 / "r" / "ref_decoded.jsonl"

RAW_W = 1296
RAW_H = 972
IMAGE_CENTER_X = RAW_W / 2.0
HALF_W = RAW_W / 2.0

FIELD3_VIDEO_PATH = VIDEO_DIR / "field3_08f_slope_push_lane_postprocess_v1.mp4"
FIELD3_SEQUENCE_CSV = TABLE_DIR / "field3_08f_slope_push_lane_postprocess_v1.csv"
CONFIG_JSON = CONFIG_DIR / "lane_behavior_08f_slope_push_v1.json"
SUMMARY_JSON = REVIEW_ROOT / "slope_push_summary_v1.json"


LANE_BEHAVIOR_08F = {
    "name": "slope_push_lane_postprocess_v1",
    "description": "No target path. Weighted local slope gives direction; nearby lanes push steer away from lane boundary.",

    # Geometry read points. They are normalized by original camera height.
    "near_y_ratio": 0.96,
    "mid_y_ratio": 0.84,
    "far_y_ratio": 0.68,

    # Lane feature filtering. These are internal robustness gates, not field tuning knobs.
    "min_points": 4,
    "min_y_span_ratio": 0.06,
    "max_extrapolate_ratio": 0.20,

    # Main field-tuning knobs.
    # slope_gain: how strongly the car follows local lane direction.
    # push_gain: how strongly it moves away when a lane is too close to image center.
    # safe_distance_ratio: center safety band width, relative to half image width.
    "slope_gain": 0.34,
    "push_gain": 0.42,
    "safe_distance_ratio": 0.38,
    "max_push": 0.75,

    # Trust weighting for slope. The final slope is a weighted average over visible lanes.
    "trust_conf_weight": 0.35,
    "trust_span_weight": 0.25,
    "trust_memory_weight": 0.30,
    "trust_center_weight": 0.10,
    "memory_slope_tolerance": 0.75,
    "slope_memory_alpha": 0.25,

    # Temporal smoothing.
    "steer_alpha": 0.48,
    "lost_steer_alpha": 0.20,

    # Lost handling. Keep last steering briefly, then decay while keeping last slope bias.
    "lost_hold_frames": 3,
    "lost_decay": 0.90,
    "lost_slope_gain": 0.08,

    # Diagnostic speed suggestion. Final speed still belongs to the state machine / motor layer.
    "min_speed_scale": 0.35,
    "risk_slowdown": 0.48,
    "low_conf_slowdown": 0.28,

    "max_steer_norm": 0.70,
}


def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def read_records_csv(path):
    with Path(path).open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def imread_bgr_unicode(path):
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img


def write_image_unicode(path, img):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
    if not ok:
        raise RuntimeError(path)
    buf.tofile(str(path))


def normalize_lane_points(lane):
    pts = np.array(lane.get("points", []), dtype=np.float32)
    if pts.ndim != 2 or pts.shape[1] != 2:
        return np.zeros((0, 2), dtype=np.float32)
    valid = (
        np.isfinite(pts[:, 0])
        & np.isfinite(pts[:, 1])
        & (pts[:, 0] >= -RAW_W * 0.25)
        & (pts[:, 0] <= RAW_W * 1.25)
        & (pts[:, 1] >= 0)
        & (pts[:, 1] <= RAW_H)
    )
    return pts[valid]


def x_at_y(points, y_query):
    pts = np.asarray(points, dtype=np.float32)
    if len(pts) < 2:
        return None
    order = np.argsort(pts[:, 1])
    ys = pts[order, 1]
    xs = pts[order, 0]
    uniq_y, uniq_idx = np.unique(ys, return_index=True)
    ys = uniq_y
    xs = xs[uniq_idx]
    if len(ys) < 2:
        return None
    yq = float(y_query)
    if ys[0] <= yq <= ys[-1]:
        return float(np.interp(yq, ys, xs))
    if yq < ys[0]:
        y0, y1 = float(ys[0]), float(ys[1])
        x0, x1 = float(xs[0]), float(xs[1])
    else:
        y0, y1 = float(ys[-2]), float(ys[-1])
        x0, x1 = float(xs[-2]), float(xs[-1])
    if abs(y1 - y0) < 1e-6:
        return x0
    return float(x0 + (x1 - x0) * ((yq - y0) / (y1 - y0)))


def lane_feature(lane, cfg):
    pts = normalize_lane_points(lane)
    if len(pts) < int(cfg["min_points"]):
        return None

    y_span = float(pts[:, 1].max() - pts[:, 1].min())
    if y_span < float(cfg["min_y_span_ratio"]) * RAW_H:
        return None

    near_y = float(cfg["near_y_ratio"]) * RAW_H
    mid_y = float(cfg["mid_y_ratio"]) * RAW_H
    far_y = float(cfg["far_y_ratio"]) * RAW_H
    query_ys = np.array([near_y, mid_y, far_y], dtype=np.float32)

    extrapolate_dist = 0.0
    min_y = float(pts[:, 1].min())
    max_y = float(pts[:, 1].max())
    for yq in query_ys:
        if yq < min_y:
            extrapolate_dist = max(extrapolate_dist, min_y - float(yq))
        elif yq > max_y:
            extrapolate_dist = max(extrapolate_dist, float(yq) - max_y)
    if extrapolate_dist > float(cfg["max_extrapolate_ratio"]) * RAW_H:
        return None

    x_near = x_at_y(pts, near_y)
    x_mid = x_at_y(pts, mid_y)
    x_far = x_at_y(pts, far_y)
    if x_near is None or x_mid is None or x_far is None:
        return None

    # Positive heading means the lane points toward image-right as y goes upward.
    heading = (float(x_far) - float(x_near)) / max(1.0, near_y - far_y)
    center_dist_ratio = (float(x_mid) - IMAGE_CENTER_X) / HALF_W
    near_dist_ratio = (float(x_near) - IMAGE_CENTER_X) / HALF_W

    inside_count = int(((query_ys >= min_y) & (query_ys <= max_y)).sum())
    coverage_score = inside_count / 3.0
    span_score = clamp(y_span / (RAW_H * 0.28), 0.0, 1.0)
    point_score = clamp(len(pts) / 22.0, 0.0, 1.0)
    conf_score = clamp(float(lane.get("conf", 0.0)) / 0.85, 0.0, 1.0)

    return {
        "points": pts,
        "conf": float(lane.get("conf", 0.0)),
        "x_near": float(x_near),
        "x_mid": float(x_mid),
        "x_far": float(x_far),
        "heading": float(heading),
        "center_dist_ratio": float(center_dist_ratio),
        "near_dist_ratio": float(near_dist_ratio),
        "coverage_score": float(coverage_score),
        "span_score": float(span_score),
        "point_score": float(point_score),
        "conf_score": float(conf_score),
    }


def extract_lane_features(lanes, cfg):
    features = []
    for lane in lanes:
        feat = lane_feature(lane, cfg)
        if feat is not None:
            features.append(feat)
    # Keep closer-to-center lanes first for drawing stability, but slope uses weights.
    features.sort(key=lambda f: abs(f["center_dist_ratio"]))
    return features


def init_memory():
    return {
        "last_slope": 0.0,
        "last_steer": 0.0,
        "lost_frames": 0,
        "seen_lane": False,
    }


def lane_trust(feat, memory, cfg):
    conf_part = feat["conf_score"]
    span_part = 0.55 * feat["span_score"] + 0.25 * feat["point_score"] + 0.20 * feat["coverage_score"]
    if memory.get("seen_lane", False):
        diff = abs(feat["heading"] - float(memory.get("last_slope", 0.0)))
        memory_part = clamp(1.0 - diff / float(cfg["memory_slope_tolerance"]), 0.0, 1.0)
    else:
        memory_part = 1.0
    # Lanes extremely close to image edge are less reliable as global direction cues.
    center_part = clamp(1.0 - max(0.0, abs(feat["center_dist_ratio"]) - 0.55) / 0.45, 0.0, 1.0)

    w = (
        float(cfg["trust_conf_weight"]) * conf_part
        + float(cfg["trust_span_weight"]) * span_part
        + float(cfg["trust_memory_weight"]) * memory_part
        + float(cfg["trust_center_weight"]) * center_part
    )
    return float(clamp(w, 0.0, 1.0))


def weighted_slope(features, memory, cfg):
    if not features:
        return float(memory.get("last_slope", 0.0)), 0.0, []

    weights = [lane_trust(f, memory, cfg) for f in features]
    total = float(sum(weights))
    if total < 1e-6:
        return float(memory.get("last_slope", 0.0)), 0.0, weights
    slope = float(sum(f["heading"] * w for f, w in zip(features, weights)) / total)
    confidence = float(clamp(total / max(1, len(features)), 0.0, 1.0))
    return slope, confidence, weights


def safety_push(features, cfg):
    safe_dist = float(cfg["safe_distance_ratio"]) * HALF_W
    push = 0.0
    risk = 0.0
    closest_abs_dx = None

    for feat in features:
        dx = float(feat["x_near"] - IMAGE_CENTER_X)
        abs_dx = abs(dx)
        closest_abs_dx = abs_dx if closest_abs_dx is None else min(closest_abs_dx, abs_dx)
        if abs_dx < safe_dist:
            intensity = 1.0 - abs_dx / max(1.0, safe_dist)
            # Lane on left -> push right; lane on right -> push left.
            if abs(dx) < 1e-6:
                sign = 0.0
            else:
                sign = -np.sign(dx)
            push += float(sign * intensity)
            risk = max(risk, float(intensity))

    push = clamp(push, -float(cfg["max_push"]), float(cfg["max_push"]))
    return float(push), float(risk), closest_abs_dx


def recommended_speed_scale(departure_risk, lane_confidence, cfg):
    speed = 1.0
    speed -= float(cfg["risk_slowdown"]) * float(departure_risk)
    speed -= float(cfg["low_conf_slowdown"]) * float(1.0 - lane_confidence)
    return float(clamp(speed, float(cfg["min_speed_scale"]), 1.0))


def update_drive_08f(lanes, memory, cfg):
    features = extract_lane_features(lanes, cfg)

    if features:
        memory["lost_frames"] = 0
        slope, lane_conf, weights = weighted_slope(features, memory, cfg)
        push, risk, closest_abs_dx = safety_push(features, cfg)

        raw_steer = float(cfg["slope_gain"]) * slope + float(cfg["push_gain"]) * push
        raw_steer = clamp(raw_steer, -float(cfg["max_steer_norm"]), float(cfg["max_steer_norm"]))
        steer = (1.0 - float(cfg["steer_alpha"])) * float(memory["last_steer"]) + float(cfg["steer_alpha"]) * raw_steer
        steer = clamp(steer, -float(cfg["max_steer_norm"]), float(cfg["max_steer_norm"]))

        memory["last_slope"] = (1.0 - float(cfg["slope_memory_alpha"])) * float(memory["last_slope"]) + float(cfg["slope_memory_alpha"]) * slope
        memory["last_steer"] = steer
        memory["seen_lane"] = True

        mode = "slope_push_risk" if risk >= 0.35 else "slope_push"
        speed_hint = recommended_speed_scale(risk, lane_conf, cfg)
        return {
            "mode": mode,
            "steer_norm": float(steer),
            "raw_steer": float(raw_steer),
            "weighted_slope": float(slope),
            "memory_slope": float(memory["last_slope"]),
            "push_term": float(push),
            "departure_risk": float(risk),
            "lane_confidence": float(lane_conf),
            "speed_hint": float(speed_hint),
            "visible_lane_count": int(len(lanes)),
            "feature_count": int(len(features)),
            "closest_abs_dx": None if closest_abs_dx is None else float(closest_abs_dx),
            "weights": weights,
            "features": features,
            "reason": "visible_lanes_weighted_slope_plus_push",
        }

    memory["lost_frames"] = int(memory.get("lost_frames", 0)) + 1
    if memory["lost_frames"] <= int(cfg["lost_hold_frames"]):
        raw_steer = float(memory["last_steer"])
        mode = "lost_hold"
        reason = "no_feature_hold_last_steer"
    else:
        raw_steer = float(memory["last_steer"]) * float(cfg["lost_decay"]) + float(cfg["lost_slope_gain"]) * float(memory["last_slope"])
        mode = "lost_slope_decay"
        reason = "no_feature_decay_with_memory_slope"

    raw_steer = clamp(raw_steer, -float(cfg["max_steer_norm"]), float(cfg["max_steer_norm"]))
    steer = (1.0 - float(cfg["lost_steer_alpha"])) * float(memory["last_steer"]) + float(cfg["lost_steer_alpha"]) * raw_steer
    steer = clamp(steer, -float(cfg["max_steer_norm"]), float(cfg["max_steer_norm"]))
    memory["last_steer"] = steer

    risk = 0.70
    lane_conf = 0.0
    return {
        "mode": mode,
        "steer_norm": float(steer),
        "raw_steer": float(raw_steer),
        "weighted_slope": float(memory.get("last_slope", 0.0)),
        "memory_slope": float(memory.get("last_slope", 0.0)),
        "push_term": 0.0,
        "departure_risk": float(risk),
        "lane_confidence": float(lane_conf),
        "speed_hint": recommended_speed_scale(risk, lane_conf, cfg),
        "visible_lane_count": int(len(lanes)),
        "feature_count": 0,
        "closest_abs_dx": None,
        "weights": [],
        "features": [],
        "reason": reason,
    }


def load_field3_sequence():
    records = read_records_csv(RECORDS_CSV)
    decoded_rows = read_jsonl(DECODED_JSONL)
    decoded_by_key = {r["key"]: r for r in decoded_rows}
    seq = [r for r in records if r["set"] == "field3" and r["role"] == "sequence"]
    seq.sort(key=lambda r: int(r["order"]))
    return seq, decoded_by_key


def draw_lane(img, lane, color=(0, 220, 0), thickness=2):
    pts = normalize_lane_points(lane).astype(np.int32)
    if len(pts) >= 2:
        cv2.polylines(img, [pts.reshape(-1, 1, 2)], False, color, thickness, cv2.LINE_AA)


def draw_arrow(img, steer, color, origin_y_ratio=0.78, length=145, thickness=4):
    h, w = img.shape[:2]
    x0 = int(w * 0.50)
    y0 = int(h * origin_y_ratio)
    x1 = int(x0 + steer * length)
    y1 = int(y0 - length * 0.70)
    cv2.arrowedLine(img, (x0, y0), (x1, y1), color, thickness, cv2.LINE_AA, tipLength=0.22)


def overlay_drive(rec, result, cfg, decoded_by_key, scale_width=960):
    img = imread_bgr_unicode(PKG10 / rec["image_rel"])
    decoded = decoded_by_key[rec["key"]]
    for lane in decoded.get("lanes", []):
        draw_lane(img, lane)

    h, w = img.shape[:2]
    center_x = int(w * 0.5)
    safe_px = int(float(cfg["safe_distance_ratio"]) * (w / 2.0))

    # Safety band: lanes entering this band produce push.
    cv2.line(img, (center_x, 0), (center_x, h - 1), (230, 230, 230), 1, cv2.LINE_AA)
    cv2.line(img, (center_x - safe_px, 0), (center_x - safe_px, h - 1), (80, 220, 255), 1, cv2.LINE_AA)
    cv2.line(img, (center_x + safe_px, 0), (center_x + safe_px, h - 1), (80, 220, 255), 1, cv2.LINE_AA)

    for feat, weight in zip(result.get("features", []), result.get("weights", [])):
        x = int(round(feat["x_near"]))
        y = int(round(float(cfg["near_y_ratio"]) * RAW_H))
        radius = max(4, int(10 * weight))
        color = (0, 160, 255) if abs(feat["x_near"] - IMAGE_CENTER_X) < safe_px else (255, 180, 0)
        cv2.circle(img, (x, y), radius, color, -1, cv2.LINE_AA)
        cv2.putText(img, f"w={weight:.2f}", (x + 6, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)

    # Thin cyan arrow is raw steer. Thick magenta arrow is smoothed final steer.
    draw_arrow(img, result["raw_steer"], (255, 255, 0), origin_y_ratio=0.82, length=125, thickness=2)
    draw_arrow(img, result["steer_norm"], (255, 0, 255), origin_y_ratio=0.84, length=150, thickness=4)

    lines = [
        f'#{int(rec["order"]):03d} {result["mode"]} lanes={result["visible_lane_count"]} feat={result["feature_count"]}',
        f'steer={result["steer_norm"]:+.2f} raw={result["raw_steer"]:+.2f} slope={result["weighted_slope"]:+.2f} push={result["push_term"]:+.2f}',
        f'risk={result["departure_risk"]:.2f} conf={result["lane_confidence"]:.2f} speed_hint={result["speed_hint"]:.2f}',
    ]
    y0 = 28
    for i, text in enumerate(lines):
        y = y0 + i * 25
        cv2.putText(img, text, (14, y), cv2.FONT_HERSHEY_SIMPLEX, 0.66, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(img, text, (14, y), cv2.FONT_HERSHEY_SIMPLEX, 0.66, (255, 255, 255), 1, cv2.LINE_AA)

    if scale_width is not None and img.shape[1] != scale_width:
        scale = scale_width / img.shape[1]
        img = cv2.resize(img, (scale_width, int(round(img.shape[0] * scale))), interpolation=cv2.INTER_AREA)
    return img


def write_video(frames, out_path, fps=12):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if not frames:
        raise RuntimeError("no frames")
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for frame in frames:
        writer.write(frame)
    writer.release()


def make_contact_sheet(rows, decoded_by_key, out_path, title, cfg, cols=3, scale_width=520):
    imgs = []
    for rec, result in rows:
        img = overlay_drive(rec, result, cfg, decoded_by_key, scale_width=scale_width)
        imgs.append(img)
    if not imgs:
        return
    h, w = imgs[0].shape[:2]
    rows_n = int(np.ceil(len(imgs) / cols))
    sheet = np.zeros((rows_n * h + 46, cols * w, 3), dtype=np.uint8)
    sheet[:] = 245
    cv2.putText(sheet, title, (16, 32), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (20, 20, 20), 2, cv2.LINE_AA)
    for idx, img in enumerate(imgs):
        r = idx // cols
        c = idx % cols
        y0 = 46 + r * h
        x0 = c * w
        sheet[y0:y0+h, x0:x0+w] = img
    write_image_unicode(out_path, sheet)


def run_field3_replay():
    seq, decoded_by_key = load_field3_sequence()
    memory = init_memory()
    rows = []
    frames = []

    for rec in seq:
        decoded = decoded_by_key[rec["key"]]
        result = update_drive_08f(decoded.get("lanes", []), memory, LANE_BEHAVIOR_08F)
        row = {
            "key": rec["key"],
            "order": int(rec["order"]),
            "source_name": rec.get("source_name", ""),
            "mode": result["mode"],
            "visible_lane_count": result["visible_lane_count"],
            "feature_count": result["feature_count"],
            "steer_norm": result["steer_norm"],
            "raw_steer": result["raw_steer"],
            "weighted_slope": result["weighted_slope"],
            "memory_slope": result["memory_slope"],
            "push_term": result["push_term"],
            "departure_risk": result["departure_risk"],
            "lane_confidence": result["lane_confidence"],
            "speed_hint": result["speed_hint"],
            "closest_abs_dx": "" if result["closest_abs_dx"] is None else result["closest_abs_dx"],
            "reason": result["reason"],
        }
        rows.append((rec, result, row))
        frames.append(overlay_drive(rec, result, LANE_BEHAVIOR_08F, decoded_by_key, scale_width=960))

    write_video(frames, FIELD3_VIDEO_PATH, fps=12)

    with FIELD3_SEQUENCE_CSV.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0][2].keys()))
        writer.writeheader()
        for _, _, row in rows:
            writer.writerow(row)

    write_json(CONFIG_JSON, LANE_BEHAVIOR_08F)

    table_rows = [row for _, _, row in rows]
    mode_counts = Counter(r["mode"] for r in table_rows)
    summary = {
        "frames": len(table_rows),
        "video": str(FIELD3_VIDEO_PATH),
        "table": str(FIELD3_SEQUENCE_CSV),
        "config": str(CONFIG_JSON),
        "mode_counts": dict(mode_counts),
        "mean_abs_steer": float(np.mean([abs(float(r["steer_norm"])) for r in table_rows])),
        "p90_abs_steer": float(np.percentile([abs(float(r["steer_norm"])) for r in table_rows], 90)),
        "max_abs_steer": float(np.max([abs(float(r["steer_norm"])) for r in table_rows])),
        "mean_departure_risk": float(np.mean([float(r["departure_risk"]) for r in table_rows])),
        "p90_departure_risk": float(np.percentile([float(r["departure_risk"]) for r in table_rows], 90)),
        "mean_lane_confidence": float(np.mean([float(r["lane_confidence"]) for r in table_rows])),
        "mean_speed_hint": float(np.mean([float(r["speed_hint"]) for r in table_rows])),
        "min_speed_hint": float(np.min([float(r["speed_hint"]) for r in table_rows])),
    }
    write_json(SUMMARY_JSON, summary)

    return rows, decoded_by_key, summary


# Mini synthetic sanity check.
memory = init_memory()
toy_lanes = [
    {"conf": 0.9, "points": [[350, 960], [390, 880], [430, 800], [500, 660]]},
    {"conf": 0.9, "points": [[950, 960], [910, 880], [870, 800], [800, 660]]},
]
toy_result = update_drive_08f(toy_lanes, memory, LANE_BEHAVIOR_08F)
print("toy_result", {k: toy_result[k] for k in ["mode", "steer_norm", "weighted_slope", "push_term", "departure_risk", "lane_confidence", "speed_hint"]})

toy_result {'mode': 'slope_push', 'steer_norm': 1.9095304180187607e-17, 'weighted_slope': 1.1700554031977699e-16, 'push_term': 0.0, 'departure_risk': 0.0, 'lane_confidence': 0.9488636363636364, 'speed_hint': 0.9856818181818182}


In [2]:
rows, decoded_by_key, summary = run_field3_replay()
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "frames": 240,
  "video": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\12_clrkdnet_supervised_rebuild\\review_outputs\\08f_slope_push_lane_postprocess_v1\\videos\\field3_08f_slope_push_lane_postprocess_v1.mp4",
  "table": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\12_clrkdnet_supervised_rebuild\\review_outputs\\08f_slope_push_lane_postprocess_v1\\tables\\field3_08f_slope_push_lane_postprocess_v1.csv",
  "config": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\12_clrkdnet_supervised_rebuild\\review_outputs\\08f_slope_push_lane_postprocess_v1\\config\\lane_behavior_08f_slope_push_v1.json",
  "mode_counts": {
    "slope_push": 185,
    "lost_hold": 35,
    "slope_push_risk": 20
  },
  "mean_abs_steer": 0.17551121321092922,
  "p90_abs_steer": 0.3398298413462777,
  "max_abs_steer": 0.5904677638679547,
  "mean_departure_risk": 0.1764009453162296,
  "p90_departu

In [3]:
# Contact sheets for quick visual judgement.
interval = []
for idx in list(range(0, len(rows), 20)):
    rec, result, _ = rows[idx]
    interval.append((rec, result))
if rows[-1][0]["key"] != interval[-1][0]["key"]:
    interval.append((rows[-1][0], rows[-1][1]))

interesting_indices = []
for i, (_, result, row) in enumerate(rows):
    if result["departure_risk"] >= 0.45 or result["mode"] != "slope_push":
        interesting_indices.append(i)
interesting_indices = sorted(set(interesting_indices[:18] + list(np.argsort([-abs(r[1]["steer_norm"]) for r in rows])[:9])))
interesting = [(rows[i][0], rows[i][1]) for i in interesting_indices[:24]]

make_contact_sheet(
    interval,
    decoded_by_key,
    FRAME_REVIEW_DIR / "interval_sheet.jpg",
    "08f interval frames",
    LANE_BEHAVIOR_08F,
    cols=3,
    scale_width=520,
)
make_contact_sheet(
    interesting,
    decoded_by_key,
    FRAME_REVIEW_DIR / "risk_and_extreme_sheet.jpg",
    "08f risk / non-normal / extreme frames",
    LANE_BEHAVIOR_08F,
    cols=3,
    scale_width=520,
)
print("wrote", FRAME_REVIEW_DIR / "interval_sheet.jpg")
print("wrote", FRAME_REVIEW_DIR / "risk_and_extreme_sheet.jpg")

wrote ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\08f_slope_push_lane_postprocess_v1\frame_review\interval_sheet.jpg
wrote ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\08f_slope_push_lane_postprocess_v1\frame_review\risk_and_extreme_sheet.jpg


## 판단 포인트

08f는 "잘 도는지"보다 먼저 아래를 확인한다.

- 노란 차선이 중앙 안전 band 안으로 들어올 때 `push_term`이 반대로 작동하는가?
- `weighted_slope`가 frame마다 과하게 튀지 않는가?
- `departure_risk`가 실제로 차선을 밟을 것 같은 frame에서 커지는가?
- `lane_confidence`가 lane이 사라지거나 짧아지는 구간에서 낮아지는가?
- state machine 입장에서는 `steer_norm`, `departure_risk`, `lane_confidence`만 보면 충분한가?

현장 튜닝 우선순위는 다음 네 개다.

```text
slope_gain:       곡선에서 방향을 얼마나 따라갈지
push_gain:        차선이 가까워질 때 얼마나 강하게 밀어낼지
safe_distance_ratio: 차선을 너무 가까움으로 볼 화면 중앙 band 폭
steer_alpha:      조향 반응성을 얼마나 빠르게 할지
```